In [2]:
import tensorflow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dropout, Dense, Bidirectional
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping


In [3]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg

[nltk_data] Downloading package gutenberg to /usr/share/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [4]:
data = gutenberg.raw('shakespeare-hamlet.txt')
text =data.lower()

text[:500]


"[the tragedie of hamlet by william shakespeare 1599]\n\n\nactus primus. scoena prima.\n\nenter barnardo and francisco two centinels.\n\n  barnardo. who's there?\n  fran. nay answer me: stand & vnfold\nyour selfe\n\n   bar. long liue the king\n\n   fran. barnardo?\n  bar. he\n\n   fran. you come most carefully vpon your houre\n\n   bar. 'tis now strook twelue, get thee to bed francisco\n\n   fran. for this releefe much thankes: 'tis bitter cold,\nand i am sicke at heart\n\n   barn. haue you had quiet guard?\n  fran. not"

In [5]:
import re
text = re.sub(r'[^a-z\n\s]', ' ', text)
text = re.sub(r'[ ]+', ' ', text)

In [6]:

lines = text.split('\n')
lines = [line.strip() for line in lines if line.strip() != ""]


lines = [re.sub(r"\[.*?\]", "", line) for line in lines]
lines = [re.sub(r"[^\w\s]", "", line) for line in lines]
lines[:5]

['the tragedie of hamlet by william shakespeare',
 'actus primus scoena prima',
 'enter barnardo and francisco two centinels',
 'barnardo who s there',
 'fran nay answer me stand vnfold']

In [7]:
train_lines ,test_lines = train_test_split(
    lines,
    test_size=0.2,
    random_state=42
)



In [8]:
##tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_lines)

total_words = len(tokenizer.word_index) + 1
total_words

4115

In [9]:
#index
tokenizer.word_index

{'the': 1,
 'and': 2,
 'to': 3,
 'of': 4,
 'i': 5,
 'my': 6,
 'you': 7,
 'a': 8,
 'it': 9,
 'in': 10,
 'that': 11,
 'ham': 12,
 'is': 13,
 'not': 14,
 'his': 15,
 'this': 16,
 'but': 17,
 'your': 18,
 'with': 19,
 'for': 20,
 'me': 21,
 'd': 22,
 'lord': 23,
 'what': 24,
 'he': 25,
 'as': 26,
 'be': 27,
 'so': 28,
 'haue': 29,
 'him': 30,
 'king': 31,
 'will': 32,
 'no': 33,
 's': 34,
 'are': 35,
 'we': 36,
 'on': 37,
 'our': 38,
 'if': 39,
 'all': 40,
 'thou': 41,
 'shall': 42,
 'or': 43,
 'hamlet': 44,
 'by': 45,
 'come': 46,
 'then': 47,
 'good': 48,
 'do': 49,
 'hor': 50,
 'let': 51,
 'now': 52,
 'her': 53,
 't': 54,
 'there': 55,
 'thy': 56,
 'from': 57,
 'more': 58,
 'how': 59,
 'was': 60,
 'oh': 61,
 'they': 62,
 'enter': 63,
 'like': 64,
 'most': 65,
 'at': 66,
 'know': 67,
 'would': 68,
 'tis': 69,
 'vs': 70,
 'well': 71,
 'them': 72,
 'selfe': 73,
 'did': 74,
 'o': 75,
 'loue': 76,
 'why': 77,
 'th': 78,
 'which': 79,
 'may': 80,
 'qu': 81,
 'giue': 82,
 'hath': 83,
 'when': 

In [10]:
def create_seq(lines):
    sequence =[]
    for line in lines:
        token_list = tokenizer.texts_to_sequences([line])[0]
        for i in range(1,len(token_list)):
            ngram_seq = token_list[:i+1]
            sequence.append(ngram_seq)

    return sequence

In [11]:
train_sequences = create_seq(train_lines)
test_sequences  = create_seq(test_lines)

In [12]:
train_sequences[:5]

[[1, 383],
 [1, 383, 35],
 [1, 383, 35, 986],
 [1, 383, 35, 986, 11],
 [1, 383, 35, 986, 11, 98]]

In [13]:

test_sequences[5:10]

[[12, 33, 33],
 [12, 33, 33, 5],
 [12, 33, 33, 5, 170],
 [12, 33, 33, 5, 170, 443],
 [12, 33, 33, 5, 170, 443, 7]]

In [14]:
max_len = max([len(x) for x in train_sequences])
max_len

14

In [15]:
train_sequences = pad_sequences(
    train_sequences,
    maxlen=max_len,
    padding='pre'
)

train_sequences[:5]

array([[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   1,
        383],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   1, 383,
         35],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   1, 383,  35,
        986],
       [  0,   0,   0,   0,   0,   0,   0,   0,   0,   1, 383,  35, 986,
         11],
       [  0,   0,   0,   0,   0,   0,   0,   0,   1, 383,  35, 986,  11,
         98]], dtype=int32)

In [16]:
test_sequences = pad_sequences(
    test_sequences,
    maxlen=max_len,
    padding='pre'
)

In [17]:
X_train = train_sequences[:,:-1]
y_train = train_sequences[:,-1]

In [18]:
X_test = test_sequences[:, :-1]
y_test = test_sequences[:, -1]

In [20]:

model = Sequential([
    Input(shape=(max_len-1,)),
    Embedding(total_words, 64),
    Bidirectional(
        LSTM(100, return_sequences=True)
    ),
    Dropout(0.2),
    Bidirectional(
        LSTM(60)
    ),
    Dense(total_words, activation="softmax")
])

In [21]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 13, 64)         │       263,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 13, 200)        │       132,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 13, 200)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 120)            │       125,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4115)           │       497,915 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,018,555 (3.89 MB)

 Trainable params: 1,018,555 (3.89 MB)

 Non-trainable params: 0 (0.00 B)

In [23]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size = 32,
    validation_data=(X_test,y_test)
)

Epoch 1/50


I0000 00:00:1773250539.481169     142 cuda_dnn.cc:529] Loaded cuDNN version 91002


661/661 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - accuracy: 0.0285 - loss: 7.0215 - val_accuracy: 0.0415 - val_loss: 6.2508
Epoch 2/50
661/661 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.0358 - loss: 6.3679 - val_accuracy: 0.0338 - val_loss: 6.1288
Epoch 3/50
661/661 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.0392 - loss: 6.2670 - val_accuracy: 0.0511 - val_loss: 6.0711
Epoch 4/50
661/661 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.0514 - loss: 6.1379 - val_accuracy: 0.0562 - val_loss: 6.0152
Epoch 5/50
661/661 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.0583 - loss: 6.0115 - val_accuracy: 0.0571 - val_loss: 5.9879
Epoch 6/50
661/661 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.0673 - loss: 5.8519 - val_accuracy: 0.0691 - val_loss: 5.9521
Epoch 7/50
661/661 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.0828 - loss: 5.6583 - val_accuracy: 0.0746 - val_loss: 5.9595
Epoch 8/50
661/661 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.0879 - loss: 5.5574 - val_accurac

In [29]:
import numpy as np
def predict_next_word(model, tokenizer, text, max_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_len:
        token_list = token_list[-(max_len-1):]  # Ensure the sequence length matches max_sequence_len-1
    token_list = pad_sequences([token_list], maxlen=max_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None

In [31]:
input_text = "Something is rotten in the state"
print(f'input is: {input_text}')
max_len = model.input_shape[1]+1
next_word = predict_next_word(model,tokenizer,input_text,max_len)
print(f"Next Word Prediction:{next_word}")

input is: Something is rotten in the state
Next Word Prediction:of


In [36]:
import pickle
model.save("next_word_lstm.h5")

with open('tokenizer.pickle','wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)